In [ ]:
import subprocess # para correr programas externos a Python (los .cpp)
import csv # para escribir archivos CSV
import os # para interactuar con el sistema operativo

In [ ]:
# parámetros del experimento
PUNTOS     = [100000, 200000, 300000, 400000, 600000, 800000, 1000000]
DIMS       = [2, 3]
HILOS      = [1, 5, 10, 20]
K          = 5
MAX_ITER   = 100
ITERACIONES = 10


SERIAL   = "./kmeans_serial"  # ruta relativa al ejecutable serial
PARALELO = "./kmeans_paralelo"  # ruta relativa al ejectuable paralelo
CARPETA_CSV = "csv"

archivo_resultados = "resultados.csv"  # aqui se guardaran los tiempos medidos por prueba

In [ ]:
with open(archivo_resultados, "w", newline="") as f:  # abre el archivo_resultados, cuando termine el bloque lo cierra
    writer = csv.writer(f)  # objeto para escribir filas en el csv
    writer.writerow(["tipo", "puntos", "dims", "hilos", "iteracion", "tiempo_ms"])  # escribe la primera fila, o sea el encabezado

def extraer_tiempo(salida):  # funcion recibe la salida en string del programa
    for linea in salida.splitlines():  # divide la salida en lineas y recorre una por una
        if "Tiempo" in linea:
            partes = linea.split(":")  # divida la linea con la palabra "tiempo" en dos partes delimitadas por el :
            tiempo_str = partes[1].replace("ms", "").strip() # agarra la segunda parte de la linea y quita el "ms" del tiempo
            return float(tiempo_str)
    return -1.0  # si no encontro linea con "Tiempo" devuelve -1.0

def correr(comando):   # recibe una lista con el comando a ejectuar. Pj: ["./kmeans_serial", "csv/100000_2d.csv", "5", "100"]
    resultado = subprocess.run(comando, capture_output=True, text=True)  # aqui se usa el subprocess que sirve para leer programas externos
    return extraer_tiempo(resultado.stdout)  # toma la salida del programa externo y extrae el tiempo

total = len(PUNTOS) * len(DIMS) * (1 + len(HILOS)) * ITERACIONES  # Calcula cuantas pruebas se van a ejecutar en total
                                                                  # se usa (1 + len(HILOS)) porque se hara 1 corrida serial y varias paralelas
corrida = 0  # lleva la cuenta de en cual prueba va

In [ ]:
with open(archivo_resultados, "a", newline="") as f:  # vuelve a abrir el archivo_resultados. "a" indica que se agregue al final sin borrar
    writer = csv.writer(f)  # objeto para escribir filas en el csv

    for n in PUNTOS:
        for dims in DIMS:  # recorre todas las combinaciones de num de puntos y dimension. Pj. 2D 100,000, 3D 100,000...

            ruta_csv = f"{CARPETA_CSV}/{n}_{dims}d.csv"  # construye el nombre del csv esperado

            if not os.path.exists(ruta_csv):  # busca el csv, si no existe, imprime advertencia
                print(f"ADVERTENCIA: no existe {ruta_csv}, saltando...")
                continue   

            for it in range(ITERACIONES):  # por cada iteración del 1 al 10
                corrida += 1  # contador de progreso
                t = correr([SERIAL, ruta_csv, str(K), str(MAX_ITER)])  # Ejecuta el programa serial
                writer.writerow(["serial", n, dims, 1, it+1, t])  # guarda una instancia (fila) en el CSV. Hilos=1 solo para que no quede vacio
                print(f"[{corrida}/{total}] serial | {n} puntos | {dims}D | iter {it+1} | {t:.2f} ms")  # Aqui imprime el progreso y el tiempo
                # por ejemplo: [1/700] serial | 100000 puntos | 2D | iter 1 | 28.11 ms

            for h in HILOS:  # hace lo mismo que en el serial, solo que itera tambien sobre HILOS
                for it in range(ITERACIONES):
                    corrida += 1
                    t = correr([PARALELO, ruta_csv, str(K), str(MAX_ITER), str(h)])
                    writer.writerow(["paralelo", n, dims, h, it+1, t])
                    print(f"[{corrida}/{total}] paralelo | {n} puntos | {dims}D | {h} hilos | iter {it+1} | {t:.2f} ms")

print(f"\nExperimento completo. Resultados en: {archivo_resultados}")

[1/700] serial | 100000 puntos | 2D | iter 1 | 28.11 ms
[2/700] serial | 100000 puntos | 2D | iter 2 | 28.38 ms
[3/700] serial | 100000 puntos | 2D | iter 3 | 28.38 ms
[4/700] serial | 100000 puntos | 2D | iter 4 | 30.70 ms
[5/700] serial | 100000 puntos | 2D | iter 5 | 31.86 ms
[6/700] serial | 100000 puntos | 2D | iter 6 | 28.57 ms
[7/700] serial | 100000 puntos | 2D | iter 7 | 29.84 ms
[8/700] serial | 100000 puntos | 2D | iter 8 | 28.42 ms
[9/700] serial | 100000 puntos | 2D | iter 9 | 29.46 ms
[10/700] serial | 100000 puntos | 2D | iter 10 | 29.92 ms
[11/700] paralelo | 100000 puntos | 2D | 1 hilos | iter 1 | 8.32 ms
[12/700] paralelo | 100000 puntos | 2D | 1 hilos | iter 2 | 8.61 ms
[13/700] paralelo | 100000 puntos | 2D | 1 hilos | iter 3 | 7.55 ms
[14/700] paralelo | 100000 puntos | 2D | 1 hilos | iter 4 | 7.74 ms
[15/700] paralelo | 100000 puntos | 2D | 1 hilos | iter 5 | 7.85 ms
[16/700] paralelo | 100000 puntos | 2D | 1 hilos | iter 6 | 17.84 ms
[17/700] paralelo | 100000 pu